# Практика · Згортка вручну

> Лекція: [lecture.html](lecture.html) · Домашнє: [homework.html](homework.html) · Тест: [quiz.html](quiz.html)

**Мережа не потрібна:** усі зображення ми малюємо самі, формулами. Жодного завантаження,
жодного файлу ззовні — тому числа в тебе на екрані будуть точнісінько такі самі, як у лекції.

Що зробимо:

1. намалюємо те саме «фото товару з оголошення» й переведемо його в сірий;
2. порахуємо згортку **руками** на квадратику 5 × 5 — усі девʼять множень;
3. напишемо власну реалізацію подвійним циклом і звіримо її з `cv2.filter2D`;
4. побачимо на числах, що `cv2.filter2D` рахує **кореляцію**, а не згортку;
5. перепишемо той самий цикл векторизовано й поміряємо, у скільки разів швидше;
6. подивимось, що робить сума коефіцієнтів ядра й що буває, коли її забути;
7. порівняємо чотири режими межі на одному й тому самому кутовому пікселі;
8. розкладемо гаусове ядро на два одновимірні проходи й порахуємо економію;
9. зробимо різкість двома способами — ядром і як «оригінал мінус розмите»;
10. напишемо власний Собель і звіримо з `cv2.Sobel`;
11. розберемо Кенні на чотири окремі кроки й звіримо з `cv2.Canny` числом.

## 0 · Що нам знадобиться

Три бібліотеки: `numpy` для масивів, `cv2` (OpenCV) для операцій із зображеннями
й `matplotlib`, щоб дивитись на результат очима. Плюс `time` — ним ми поміряємо,
наскільки повільний чесний подвійний цикл.

In [ ]:
import time

import numpy as np
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

print("numpy      ", np.__version__)
print("opencv     ", cv2.__version__)

## 1 · Малюємо те саме фото товару

Генератор — той самий, що в темі 01: фон стільниці, мʼяка тінь, корпус зі скругленими
кутами, екран із заставкою, зелений індикатор, відблиск на склі й шум матриці.
Шум тут не прикраса, а робоча деталь: саме через нього далі пороги Кенні
щось означатимуть.

In [ ]:
IMAGE_HEIGHT, IMAGE_WIDTH = 240, 320


def rounded_gap(row_grid, col_grid, left, top, right, bottom, radius):
    """Квадрат відстані від пікселя до скругленого прямокутника.

    Усередині прямокутника дає 0, тому одна й та сама функція годиться
    і щоб заповнити фігуру, і щоб намалювати мʼяку тінь навколо неї.
    """
    gap_x = np.maximum(np.maximum(left + radius - col_grid,
                                  col_grid - (right - radius)), 0.0)
    gap_y = np.maximum(np.maximum(top + radius - row_grid,
                                  row_grid - (bottom - radius)), 0.0)
    return gap_x * gap_x + gap_y * gap_y


def sensor_noise(height, width):
    """Детермінований «шум матриці»: у всіх читачів однаковий до байта."""
    index = np.arange(height * width * 3, dtype=np.int64)
    value = (index + 1) * 16807 % 2147483647
    value = value ^ (value >> 13)
    value = value * 48271 % 2147483647
    value = value ^ (value >> 17)
    value = value * 16807 % 2147483647
    return (value % 11 - 5).reshape(height, width, 3).astype(np.float64)


def make_phone_photo():
    """Синтетичне фото телефона: масив (240, 320, 3) типу uint8, порядок каналів RGB."""
    row_grid, col_grid = np.mgrid[0:IMAGE_HEIGHT, 0:IMAGE_WIDTH].astype(np.float64)
    down = row_grid / IMAGE_HEIGHT
    right = col_grid / IMAGE_WIDTH

    photo = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH, 3), dtype=np.float64)

    # стільниця: тепла бежева поверхня, трохи темніша знизу
    photo[:, :, 0] = 214.0 - 26.0 * down + 10.0 * right
    photo[:, :, 1] = 201.0 - 24.0 * down + 8.0 * right
    photo[:, :, 2] = 182.0 - 20.0 * down + 6.0 * right

    # тінь: силует корпусу, зсунутий вниз-вправо і розмитий по відстані
    distance = np.sqrt(rounded_gap(row_grid, col_grid, 107, 37, 235, 229, 18.0)) - 18.0
    darkness = np.clip((18.0 - distance) / 18.0, 0.0, 1.0)
    photo *= (1.0 - 0.30 * darkness)[:, :, None]

    # корпус телефона
    body = rounded_gap(row_grid, col_grid, 96, 24, 224, 216, 18.0) <= 18.0 ** 2
    sheen = 10.0 * (1.0 - down)
    photo[:, :, 0] = np.where(body, 58.0 + sheen, photo[:, :, 0])
    photo[:, :, 1] = np.where(body, 64.0 + sheen, photo[:, :, 1])
    photo[:, :, 2] = np.where(body, 74.0 + sheen, photo[:, :, 2])

    # екран: заставка з діагональним переходом від синього до помаранчевого
    screen = rounded_gap(row_grid, col_grid, 105, 33, 215, 207, 6.0) <= 6.0 ** 2
    ramp = ((col_grid - 105.0) / 110.0 + (row_grid - 33.0) / 174.0) / 2.0
    photo[:, :, 0] = np.where(screen, 26.0 + 206.0 * ramp, photo[:, :, 0])
    photo[:, :, 1] = np.where(screen, 58.0 + 66.0 * ramp, photo[:, :, 1])
    photo[:, :, 2] = np.where(screen, 170.0 - 128.0 * ramp, photo[:, :, 2])

    # зелена смужка індикатора на екрані
    indicator = rounded_gap(row_grid, col_grid, 117, 170, 203, 186, 5.0) <= 5.0 ** 2
    photo[:, :, 0] = np.where(indicator, 40.0, photo[:, :, 0])
    photo[:, :, 1] = np.where(indicator, 200.0, photo[:, :, 1])
    photo[:, :, 2] = np.where(indicator, 90.0, photo[:, :, 2])

    # відблиск на склі: світла смуга під кутом, тільки в межах екрана
    band = (col_grid - 105.0) * 0.80 + (row_grid - 33.0) * 0.55
    glare = np.clip(1.0 - np.abs(band - 74.0) / 30.0, 0.0, 1.0)
    photo += (glare * glare * 95.0 * screen)[:, :, None]

    photo += sensor_noise(IMAGE_HEIGHT, IMAGE_WIDTH)
    return np.clip(photo, 0, 255).astype(np.uint8)


photo = make_phone_photo()

# згортка живе на одному числі в пікселі, тому далі всюди працюємо із сірим
photo_bgr = photo[:, :, ::-1].copy()          # OpenCV чекає порядок каналів BGR
gray = cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2GRAY)

print("кольорове :", photo.shape, photo.dtype)
print("сіре      :", gray.shape, gray.dtype)
print("середня яскравість сірого:", round(float(gray.mean()), 2))

Дивимось очима, що саме ми обробляємо. Праворуч — те саме зображення в градаціях сірого:
одне число на піксель замість трьох.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].imshow(photo)
axes[0].set_title("фото товару з оголошення")
axes[1].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("те саме в градаціях сірого")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("далі всі операції — над сірим масивом", gray.shape)

## 2 · Рахуємо руками на квадратику 5 × 5

Перш ніж писати код, зробімо одну згортку олівцем. Візьмемо маленьке зображення 5 × 5
із різким переходом посередині — ліві три колонки темні, праві дві світлі — і ядро
розмиття 3 × 3, у якому всі девʼять коефіцієнтів однакові й дорівнюють ⅑.

Вікно 3 × 3 має де стояти лише в девʼяти положеннях: його центр не може вийти за межі.
Тому з масиву 5 × 5 виходить результат 3 × 3.

In [ ]:
tiny_image = np.array([
    [12, 14, 15, 90, 92],
    [13, 15, 16, 91, 93],
    [14, 16, 18, 92, 94],
    [15, 17, 19, 93, 95],
    [16, 18, 20, 94, 96],
], dtype=np.float64)

box_kernel = np.ones((3, 3), dtype=np.float64) / 9.0

print("зображення 5 × 5:")
print(tiny_image.astype(int))
print()
print("ядро 3 × 3 (кожен коефіцієнт = 1/9):")
print(np.round(box_kernel, 4))
print()
print("сума коефіцієнтів ядра:", round(float(box_kernel.sum()), 6))

Перше положення вікна: лівий верхній кут. Друкуємо всі девʼять множень окремо —
саме так, як їх довелося б виписати на папері.

In [ ]:
window = tiny_image[0:3, 0:3]

total = 0.0
for row in range(3):
    for col in range(3):
        product = window[row, col] * box_kernel[row, col]
        total += product
        print(f"  {window[row, col]:5.0f} × {box_kernel[row, col]:.4f} = {product:7.4f}")

print()
print("сума девʼяти доданків:", round(total, 4))
print("це число стає значенням центрального пікселя вікна — тобто клітинки (1, 1)")

Тепер те саме для всіх девʼяти положень вікна. Зверни увагу на середню колонку
результату: там число стрибає з 15 до 41, бо саме туди потрапила межа між темним
і світлим. Розмиття межу не знищило — воно її **розмазало** на три колонки.

In [ ]:
tiny_result = np.zeros((3, 3), dtype=np.float64)

for row in range(3):
    for col in range(3):
        window = tiny_image[row:row + 3, col:col + 3]
        # поелементний добуток, потім сума — це і є одна клітинка результату
        tiny_result[row, col] = np.sum(window * box_kernel)

print("результат 3 × 3, порахований руками:")
print(np.round(tiny_result, 2))
print()
print("ліва колонка — чиста темна ділянка, права — чиста світла,")
print("а середня показує, що межа розмазалась:", round(float(tiny_result[0, 1]), 2))

## 3 · Власна реалізація подвійним циклом

Тепер запишемо те саме як функцію. Вона робить рівно чотири речі:

1. добудовує навколо зображення кадр завширшки в половину ядра — щоб вікно мало де стояти
   навіть у кутовому пікселі;
2. ставить вікно на кожен піксель;
3. множить вікно на ядро поелементно й додає девʼять чисел;
4. кладе суму в центр вікна.

Друга функція, `convolve_by_hand`, відрізняється **одним рядком**: перед роботою вона
перевертає ядро на 180°. Навіщо — побачимо за два кроки.

In [ ]:
def correlate_by_hand(image, kernel, border=cv2.BORDER_REFLECT_101):
    """Кореляція подвійним циклом: вікно ковзає, множиться поелементно, сума йде в центр."""
    image = image.astype(np.float64)
    kernel = kernel.astype(np.float64)
    kernel_height, kernel_width = kernel.shape
    pad_y, pad_x = kernel_height // 2, kernel_width // 2

    # без кадру вікно не мало б де стояти в крайніх пікселях
    padded = cv2.copyMakeBorder(image, pad_y, pad_y, pad_x, pad_x, border)

    height, width = image.shape
    result = np.zeros((height, width), dtype=np.float64)
    for row in range(height):
        for col in range(width):
            window = padded[row:row + kernel_height, col:col + kernel_width]
            result[row, col] = np.sum(window * kernel)
    return result


def convolve_by_hand(image, kernel, border=cv2.BORDER_REFLECT_101):
    """Справжня згортка = кореляція з ядром, повернутим на 180°."""
    return correlate_by_hand(image, np.flip(kernel), border)


print("обидві функції готові")

Подвійний цикл на 76 800 пікселів — це 76 800 разів по девʼять множень **на рівні
Python**, а не всередині numpy. Це повільно, і ховати цього не варто. Тому беремо
**маленький фрагмент** 60 × 80 із межі корпусу телефона й міряємо час чесно.

In [ ]:
PATCH_ROW, PATCH_COL = 60, 80
patch = gray[PATCH_ROW:PATCH_ROW + 60, PATCH_COL:PATCH_COL + 80]

start = time.time()
my_blur = correlate_by_hand(patch, box_kernel)
loop_seconds = time.time() - start

print("фрагмент      :", patch.shape, "=", patch.size, "пікселів")
print("подвійний цикл:", round(loop_seconds, 3), "с")
print("на все фото це зайняло б приблизно:",
      round(loop_seconds * gray.size / patch.size, 1), "с")

## 4 · Звірка з бібліотекою

Головна перевірка курсу: наша реалізація має збігтися з бібліотечною. Симетричне ядро
розмиття не розрізняє кореляцію й згортку, тому тут збіг має бути точним.

In [ ]:
library_blur = cv2.filter2D(patch.astype(np.float64), -1, box_kernel,
                            borderType=cv2.BORDER_REFLECT_101)

difference = np.abs(my_blur - library_blur).max()
assert np.allclose(my_blur, library_blur), "наш цикл розійшовся з cv2.filter2D!"
print("✅ збігається: найбільша різниця", difference)
print("   (це не нуль рівно, а похибка чисел із рухомою комою — 10 у мінус 14 степені)")

## 5 · Найважливіше: `cv2.filter2D` рахує кореляцію, а не згортку

Операція, якою зроблені всі сучасні моделі зору, зветься **згорткою**. Але функція,
якою її рахують, згортки не рахує. Різниця в одному: справжня згортка перед роботою
**перевертає ядро на 180°**, кореляція — ні.

Симетричне ядро від повороту не змінюється, тому на розмитті різниці не видно взагалі.
Візьмемо навмисно несиметричне ядро — «змазування вправо», де коефіцієнти в рядку
ростуть зліва направо, — і подивимось на числа.

In [ ]:
motion_kernel = np.array([
    [0, 0, 0],
    [1, 2, 3],
    [0, 0, 0],
], dtype=np.float64) / 6.0

print("ядро як є:")
print(np.round(motion_kernel * 6, 0).astype(int))
print()
print("те саме ядро, повернуте на 180°:")
print(np.round(np.flip(motion_kernel) * 6, 0).astype(int))

In [ ]:
by_correlation = correlate_by_hand(patch, motion_kernel)
by_convolution = convolve_by_hand(patch, motion_kernel)
by_library = cv2.filter2D(patch.astype(np.float64), -1, motion_kernel,
                          borderType=cv2.BORDER_REFLECT_101)

print("cv2.filter2D == наша кореляція :", np.allclose(by_correlation, by_library))
print("cv2.filter2D == наша згортка   :", np.allclose(by_convolution, by_library))
print()
print("найбільша розбіжність кореляції та згортки:",
      round(float(np.abs(by_correlation - by_convolution).max()), 2), "рівнів яскравості")
print("частка пікселів, де вони різняться більш ніж на 1:",
      round(float((np.abs(by_correlation - by_convolution) > 1).mean()) * 100, 1), "%")

assert np.allclose(by_correlation, by_library), "filter2D мала збігтися саме з кореляцією"
assert not np.allclose(by_convolution, by_library), "а зі згорткою — ні"
print()
print("✅ підтверджено: cv2.filter2D — це кореляція")

Ту саму річ видно найкоротшим дослідом: подамо на вхід чорне зображення з **однією
білою точкою** посередині. Кореляція покладе в результат ядро, повернуте на 180°;
згортка — ядро як є.

Це найнадійніший спосіб перевірки. Якщо звіряти відгук із самим ядром «на око»,
дуже легко зробити протилежний висновок.

In [ ]:
impulse = np.zeros((7, 7), dtype=np.float64)
impulse[3, 3] = 6.0                      # 6, щоб коефіцієнти ядра вийшли цілими

response = cv2.filter2D(impulse, -1, motion_kernel, borderType=cv2.BORDER_CONSTANT)

print("ядро як є (× 6):")
print(np.round(motion_kernel * 6).astype(int))
print()
print("відгук cv2.filter2D на одиничний імпульс:")
print(np.round(response[2:5, 2:5]).astype(int))
print()
print("збігається з ядром як є        :", np.allclose(response[2:5, 2:5], motion_kernel * 6))
print("збігається з ядром на 180°     :", np.allclose(response[2:5, 2:5],
                                                     np.flip(motion_kernel) * 6))

**Чому це важливо і чому це не катастрофа.** Поки ядро пишеш ти рукою, різниця реальна:
несиметричне ядро дасть дзеркальний результат, і смуга змазування ляже не в той бік.
А коли ядро не пишуть, а навчають — мережа однаково вивчить ту саму матрицю в потрібному
повороті, і результат буде той самий. Тому назва «згортковий шар» прижилась, хоча
всередині рахується кореляція.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].imshow(patch, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("фрагмент як є")
axes[1].imshow(by_correlation, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("кореляція (cv2.filter2D)")
axes[2].imshow(by_convolution, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("справжня згортка")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("змазування в двох правих картинках спрямоване в різні боки")

## 6 · Той самий цикл, тільки векторизований

Подвійний цикл чесний, але повільний. Є спосіб порахувати те саме без жодного циклу
по пікселях: замість того щоб рухати вікно по зображенню, будемо рухати **зображення**
під нерухомим коефіцієнтом ядра.

Ідея в одному реченні: результат — це сума девʼяти зсунутих копій зображення, кожна
помножена на свій коефіцієнт. Циклів лишається девʼять — по клітинках ядра, а не по
76 800 пікселях.

In [ ]:
def correlate_vectorized(image, kernel, border=cv2.BORDER_REFLECT_101):
    """Та сама кореляція: сума зсунутих копій зображення, помножених на коефіцієнти."""
    image = image.astype(np.float64)
    kernel = kernel.astype(np.float64)
    kernel_height, kernel_width = kernel.shape
    pad_y, pad_x = kernel_height // 2, kernel_width // 2
    padded = cv2.copyMakeBorder(image, pad_y, pad_y, pad_x, pad_x, border)

    height, width = image.shape
    result = np.zeros((height, width), dtype=np.float64)
    for i in range(kernel_height):
        for j in range(kernel_width):
            # зсунута копія зображення того самого розміру, що й результат
            result += kernel[i, j] * padded[i:i + height, j:j + width]
    return result


start = time.time()
vector_blur = correlate_vectorized(patch, box_kernel)
vector_seconds = time.time() - start

assert np.allclose(vector_blur, my_blur), "векторизований варіант мав дати те саме!"
print("✅ той самий результат")
print("подвійний цикл :", round(loop_seconds, 4), "с")
print("векторизовано  :", round(vector_seconds, 5), "с")
print("прискорення    :", round(loop_seconds / vector_seconds), "разів")

In [ ]:
start = time.time()
full_blur = correlate_vectorized(gray, box_kernel)
print("те саме на всьому фото 240 × 320:", round(time.time() - start, 4), "с")
print("а подвійний цикл витратив би близько",
      round(loop_seconds * gray.size / patch.size, 1), "с")

## 7 · Сума коефіцієнтів ядра

Просте правило, за яким одразу видно, що ядро робить:

- сума **1** — ядро перерозподіляє яскравість, не міняючи загальної. Розмиття, різкість;
- сума **0** — ядро шукає різницю між сусідами. На рівній ділянці дає нуль,
  спалахує лише на межах. Собель, лапласіан;
- сума **інша** — картинка поїде по яскравості, і це майже завжди помилка.

Перевіримо це на числах.

In [ ]:
sharpen_kernel = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]], dtype=np.float64)
sobel_x_kernel = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]], dtype=np.float64)
emboss_kernel = np.array([[-2, -1, 0],
                          [-1, 1, 1],
                          [0, 1, 2]], dtype=np.float64)

print(f"{'ядро':<12}{'сума':>7}{'середнє входу':>16}{'середнє виходу':>17}")
for name, kernel in [("розмиття", box_kernel), ("різкість", sharpen_kernel),
                     ("Собель x", sobel_x_kernel), ("тиснення", emboss_kernel)]:
    filtered = cv2.filter2D(gray.astype(np.float64), -1, kernel,
                            borderType=cv2.BORDER_REFLECT_101)
    print(f"{name:<12}{kernel.sum():>7.2f}{gray.mean():>16.2f}{filtered.mean():>17.2f}")

А тепер типова помилка новачка: узяти бокс-фільтр і забути поділити на девʼять.

In [ ]:
forgotten = cv2.filter2D(gray.astype(np.float64), -1, np.ones((3, 3)),
                         borderType=cv2.BORDER_REFLECT_101)

print("сума коефіцієнтів:", 9.0)
print("середнє виходу   :", round(float(forgotten.mean()), 1),
      "замість", round(float(gray.mean()), 1))
print("частка чисел, що вилетіли за 255:",
      round(float((forgotten > 255).mean()) * 100, 1), "%")
print()
print("після обрізання по 255 від картинки лишиться біла пляма:")
print("унікальних значень:", len(np.unique(np.clip(forgotten, 0, 255).astype(np.uint8))))

## 8 · Краї зображення: чотири відповіді на одне питання

Вікно 3 × 3 не має де стояти в кутовому пікселі: третина його клітинок висить за межами
кадру. Відповідей три — обрізати результат, добудувати кадр, добудувати й обрізати —
а от способів **чим саме** добудувати кадр в OpenCV чотири.

Спершу подивимось, що кожен режим підставляє, на найменшому можливому прикладі:
масив із чисел від 0 до 8, до якого дописано кадр в один піксель.

In [ ]:
small = np.arange(9).reshape(3, 3).astype(np.uint8)
print("вихідний масив 3 × 3:")
print(small)
print()

border_modes = [
    ("BORDER_CONSTANT", cv2.BORDER_CONSTANT),
    ("BORDER_REPLICATE", cv2.BORDER_REPLICATE),
    ("BORDER_REFLECT", cv2.BORDER_REFLECT),
    ("BORDER_REFLECT_101", cv2.BORDER_REFLECT_101),
]

for name, mode in border_modes:
    framed = cv2.copyMakeBorder(small, 1, 1, 1, 1, mode, value=0)
    print(f"{name:<20} верхній рядок: {framed[0]}")

`BORDER_REFLECT` дублює крайній піксель (`0 0 1 2 2`), `BORDER_REFLECT_101` — ні,
він відбиває **від** нього, і тому верхній рядок починається значеннями з другого рядка
(`4 3 4 5 4`). Різниця в один піксель, а наслідки видно одразу.

Тепер те саме на справжньому зображенні: порахуємо гаусове розмиття й подивимось
на один-єдиний піксель — лівий верхній кут.

In [ ]:
gauss_1d = cv2.getGaussianKernel(9, 1.6)      # стовпчик 9 × 1
gauss_2d = gauss_1d @ gauss_1d.T              # ядро 9 × 9

print("яскравість кутового пікселя до фільтрації:", gray[0, 0])
print()
print(f"{'режим межі':<22}{'кут (0,0)':>12}{'центр (120,160)':>18}")
for name, mode in border_modes:
    blurred = cv2.filter2D(gray.astype(np.float64), -1, gauss_2d, borderType=mode)
    print(f"{name:<22}{blurred[0, 0]:>12.2f}{blurred[120, 160]:>18.2f}")

Центральний піксель однаковий у всіх чотирьох режимах — до нього кадр не дотягується.
А от кут відрізняється **утричі**: `BORDER_CONSTANT` домішує чорноту, якої в кадрі
не було, і кут темніє з ~203 до ~79. Ось чому режим межі — не дрібниця: він створює
на краю зображення структуру, якої там немає, і детектор меж її потім чесно знайде.

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for axis, (name, mode) in zip(axes, border_modes):
    blurred = cv2.filter2D(gray.astype(np.float64), -1, gauss_2d, borderType=mode)
    axis.imshow(blurred[:60, :60], cmap="gray", vmin=0, vmax=255)
    axis.set_title(name.replace("BORDER_", ""), fontsize=10)
    axis.axis("off")
plt.tight_layout()
plt.show()

print("показано лівий верхній кут 60 × 60: у першому режимі видно темну рамку")

## 9 · Бокс проти гауса й роздільність

Бокс-фільтр усереднює квадрат однаково — і сусіда збоку, і сусіда по діагоналі, який
насправді на 41 % далі. Через це точка розмазується у **квадрат**, а не в кружечок.

Поміряємо це числом: яка частка ваги ядра лежить у кутах, поза колом, вписаним у ядро.

In [ ]:
def corner_weight_share(kernel):
    """Частка ваги ядра поза колом, вписаним у його квадрат — міра «квадратності»."""
    size = kernel.shape[0]
    center = (size - 1) / 2.0
    rows, cols = np.mgrid[0:size, 0:size]
    distance = np.hypot(rows - center, cols - center)
    return kernel[distance > center].sum() / kernel.sum()


print(f"{'sigma':>6}{'розмір':>8}{'гаус':>10}{'бокс':>10}")
for sigma in (1.0, 1.5, 2.0, 3.0):
    size = int(2 * round(3 * sigma) + 1)
    gauss_kernel = cv2.getGaussianKernel(size, sigma) @ cv2.getGaussianKernel(size, sigma).T
    box = np.ones((size, size)) / (size * size)
    print(f"{sigma:>6.1f}{size:>8}{corner_weight_share(gauss_kernel) * 100:>9.2f}%"
          f"{corner_weight_share(box) * 100:>9.2f}%")

У гауса в кутах лишається один-три відсотки ваги, у бокса — третина. Саме ця третина
й малює «квадратні» сліди навколо яскравих точок.

Тепер головна практична властивість гауса: він **роздільний**. Ядро 9 × 9 розкладається
на два одновимірні проходи — спершу по рядках, потім по стовпцях, — і результат той самий.

In [ ]:
two_dimensional = cv2.filter2D(gray.astype(np.float64), -1, gauss_2d,
                               borderType=cv2.BORDER_REFLECT_101)
separable = cv2.sepFilter2D(gray.astype(np.float64), -1, gauss_1d, gauss_1d,
                            borderType=cv2.BORDER_REFLECT_101)

assert np.allclose(two_dimensional, separable, atol=1e-3), "проходи мали дати те саме!"
print("✅ два одновимірні проходи = одне двовимірне ядро")
print("найбільша різниця:", float(np.abs(two_dimensional - separable).max()))
print()
print("множень на піксель у 2D-ядра :", 9 * 9)
print("множень на піксель у двох проходів:", 9 + 9)
print("менше в", (9 * 9) / (9 + 9), "раза")

In [ ]:
gray_float = gray.astype(np.float32)
gauss_2d_f = gauss_2d.astype(np.float32)
gauss_1d_f = gauss_1d.astype(np.float32)

start = time.time()
for _ in range(200):
    cv2.filter2D(gray_float, -1, gauss_2d_f)
seconds_2d = time.time() - start

start = time.time()
for _ in range(200):
    cv2.sepFilter2D(gray_float, -1, gauss_1d_f, gauss_1d_f)
seconds_sep = time.time() - start

print("200 проходів двовимірним ядром:", round(seconds_2d, 3), "с")
print("200 проходів двома одновимірними:", round(seconds_sep, 3), "с")
print("реальне прискорення:", round(seconds_2d / seconds_sep, 2), "раза")
print()
print("теоретична економія була 4.5 — на практиці виходить приблизно стільки ж")

## 10 · Різкість: відняти розмите

«Додати різкості» насправді означає «підкреслити те, що зникає при розмитті».
Формула одна:

```
різке = оригінал + сила × (оригінал − розмите)
```

Різниця «оригінал мінус розмите» — це саме дрібні деталі й межі. Додаючи її назад,
ми робимо перепади крутішими. Цей прийом зветься **unsharp mask** — «маска нерізкості»,
і назва спантеличує: різкість роблять через **не**різку копію.

In [ ]:
blurred_for_mask = cv2.GaussianBlur(gray, (0, 0), 1.5)
detail = gray.astype(np.float64) - blurred_for_mask.astype(np.float64)

unsharp = np.clip(gray.astype(np.float64) + 1.0 * detail, 0, 255)
by_kernel = np.clip(cv2.filter2D(gray.astype(np.float64), -1, sharpen_kernel,
                                 borderType=cv2.BORDER_REFLECT_101), 0, 255)

print("стандартне відхилення яскравості (чим більше, тим контрастніше):")
print("  оригінал        :", round(float(gray.std()), 2))
print("  ядро різкості   :", round(float(by_kernel.std()), 2))
print("  unsharp mask    :", round(float(unsharp.std()), 2))
print()
print("середнє в деталях:", round(float(detail.mean()), 3),
      "— різниця «оригінал мінус розмите» майже нульова в середньому,")
print("бо на рівних ділянках розмиття нічого не міняє")

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(13, 2.9))
axes[0].imshow(gray[30:150, 90:230], cmap="gray", vmin=0, vmax=255)
axes[0].set_title("оригінал")
axes[1].imshow(blurred_for_mask[30:150, 90:230], cmap="gray", vmin=0, vmax=255)
axes[1].set_title("розмите, sigma = 1.5")
axes[2].imshow(detail[30:150, 90:230], cmap="gray", vmin=-30, vmax=30)
axes[2].set_title("деталі: оригінал − розмите")
axes[3].imshow(unsharp[30:150, 90:230], cmap="gray", vmin=0, vmax=255)
axes[3].set_title("оригінал + деталі")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("третя картинка сіра всюди, крім меж — там і живе вся різкість")

## 11 · Власний Собель проти `cv2.Sobel`

Ядро Собеля по горизонталі рахує різницю між правою й лівою колонками вікна,
даючи середньому рядку подвійну вагу — це вбудоване згладжування поперек напрямку.

Спершу порахуємо руками одну відповідь на штучній сходинці 10 → 90.

In [ ]:
step_window = np.array([[10, 10, 90],
                        [10, 10, 90],
                        [10, 10, 90]], dtype=np.float64)

print("вікно на сходинці:")
print(step_window.astype(int))
print()
print("ядро Собеля по x:")
print(sobel_x_kernel.astype(int))
print()

total = 0.0
for row in range(3):
    line = []
    for col in range(3):
        product = step_window[row, col] * sobel_x_kernel[row, col]
        total += product
        line.append(f"{step_window[row, col]:.0f}×{sobel_x_kernel[row, col]:+.0f}={product:+.0f}")
    print("  " + "   ".join(line))

print()
print("сума:", total, "= 4 × (90 − 10) = 4 × 80")

Тепер те саме на всьому фото — і звірка з бібліотекою. Тут збіг має бути **точним**,
без жодного допуску: `cv2.Sobel` із `ksize=3` — це буквально `filter2D` із цим ядром.

In [ ]:
sobel_y_kernel = np.array([[-1, -2, -1],
                           [0, 0, 0],
                           [1, 2, 1]], dtype=np.float64)

my_gx = correlate_vectorized(gray, sobel_x_kernel)
my_gy = correlate_vectorized(gray, sobel_y_kernel)

library_gx = cv2.Sobel(gray.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3,
                       borderType=cv2.BORDER_REFLECT_101)
library_gy = cv2.Sobel(gray.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3,
                       borderType=cv2.BORDER_REFLECT_101)

assert np.array_equal(my_gx, library_gx), "наш Собель по x розійшовся з cv2.Sobel!"
assert np.array_equal(my_gy, library_gy), "наш Собель по y розійшовся з cv2.Sobel!"
print("✅ збігається побітово: різниця по x", np.abs(my_gx - library_gx).max(),
      ", по y", np.abs(my_gy - library_gy).max())

Дві похідні дають дві числові характеристики кожного пікселя: **модуль** градієнта
(наскільки різкий перепад) і **напрямок** (у який бік яскравість росте найшвидше).

In [ ]:
magnitude = np.hypot(my_gx, my_gy)                    # довжина вектора (gx, gy)
direction = np.rad2deg(np.arctan2(my_gy, my_gx))      # кут у градусах, від -180 до 180

print("модуль градієнта: макс", round(float(magnitude.max()), 1),
      " медіана", round(float(np.median(magnitude)), 1))
print("на рівній ділянці стільниці (рядок 10, колонка 10):",
      round(float(magnitude[10, 10]), 1))
print("на межі корпусу (рядок 120, колонка 96):",
      round(float(magnitude[120, 96]), 1))
print()
print("напрямок у тій самій точці межі:", round(float(direction[120, 96])), "°")

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(13, 2.9))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("оригінал")
axes[1].imshow(my_gx, cmap="gray", vmin=-200, vmax=200)
axes[1].set_title("gx: вертикальні межі")
axes[2].imshow(my_gy, cmap="gray", vmin=-200, vmax=200)
axes[2].set_title("gy: горизонтальні межі")
axes[3].imshow(magnitude, cmap="magma", vmin=0, vmax=300)
axes[3].set_title("модуль: усі межі")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("верхня й нижня межі корпусу зникають на gx, бічні — на gy")

Це і є доказ, що одного ядра замало. Порахуємо, наскільки саме кожне ядро сліпе
до «чужої» межі: візьмемо вертикальну межу корпусу й горизонтальну.

In [ ]:
# вертикальна межа: ліва грань корпусу, колонка 96, рядки 60…180
vertical_edge_gx = np.abs(my_gx[60:180, 94:99]).mean()
vertical_edge_gy = np.abs(my_gy[60:180, 94:99]).mean()

# горизонтальна межа: верхня грань корпусу, рядок 24, колонки 120…200
horizontal_edge_gx = np.abs(my_gx[22:27, 120:200]).mean()
horizontal_edge_gy = np.abs(my_gy[22:27, 120:200]).mean()

print(f"{'межа':<22}{'|gx| у середньому':>20}{'|gy| у середньому':>20}")
print(f"{'вертикальна':<22}{vertical_edge_gx:>20.1f}{vertical_edge_gy:>20.1f}")
print(f"{'горизонтальна':<22}{horizontal_edge_gx:>20.1f}{horizontal_edge_gy:>20.1f}")
print()
print("ядро gx бачить вертикальну межу в",
      round(vertical_edge_gx / max(horizontal_edge_gx, 1e-9), 1),
      "раза сильніше, ніж горизонтальну")

## 12 · Кенні по кроках

Кенні — це не одна операція, а чотири, склеєні в один виклик. Розберемо їх окремо,
кожен крок своєю клітинкою.

**Крок 1 — згладити.** Похідна підсилює шум: різниця сусідів на рівній ділянці і так
скаче, а Собель цей скок ще й помножить. Тому перед градієнтом зображення розмивають.

Одразу важлива деталь, про яку мовчать підручники: **`cv2.Canny` цього кроку не робить**.
Вона починає з градієнта, а згладжувати має ти сам. Тому далі ми звірятимемо свій
результат саме з `cv2.Canny`, якій подали вже згладжене зображення.

In [ ]:
smoothed = cv2.GaussianBlur(gray, (5, 5), 1.4)

noise_before = float(gray[10:40, 10:40].std())
noise_after = float(smoothed[10:40, 10:40].std())
print("шум на рівній ділянці стільниці до згладжування :", round(noise_before, 2))
print("після                                          :", round(noise_after, 2))
print("зменшився в", round(noise_before / noise_after, 1), "раза")

**Крок 2 — градієнт.** Той самий Собель, що вище: модуль і напрямок у кожному пікселі.

In [ ]:
canny_gx = cv2.Sobel(smoothed.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3,
                     borderType=cv2.BORDER_REPLICATE)
canny_gy = cv2.Sobel(smoothed.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3,
                     borderType=cv2.BORDER_REPLICATE)
canny_magnitude = np.hypot(canny_gx, canny_gy)
canny_angle = np.rad2deg(np.arctan2(canny_gy, canny_gx)) % 180

print("модуль градієнта: макс", round(float(canny_magnitude.max()), 1))
print("пікселів, де модуль більший за 100:",
      int((canny_magnitude > 100).sum()), "— це вже занадто товсті межі")

**Крок 3 — притлумити немаксимуми.** Межа після Собеля виходить завширшки в кілька
пікселів: градієнт великий не лише точно на переході, а й поруч. Треба лишити тільки
гребінь. Правило: піксель виживає, якщо він **не менший за двох сусідів уздовж напрямку
градієнта** — тобто впоперек самої межі.

Напрямок округлюємо до одного з чотирьох: горизонталь, вертикаль і дві діагоналі.

⚠️ Тут ховається пастка, на якій спотикаються всі. У зображенні номер рядка росте
**вниз**, а не вгору, як на шкільному кресленні. Тому кут 45°, порахований через
`arctan2(gy, gx)`, вказує вправо-**вниз**, і сусіди для нього — `(row-1, col-1)` та
`(row+1, col+1)`, а не навпаки. Переплутавши дві діагоналі місцями, ти отримаєш
код, який виглядає правильним, працює без помилок і мовчки викидає половину
діагональних меж.

In [ ]:
def non_maximum_suppression(magnitude, angle_degrees):
    """Лишає тільки гребінь межі: піксель виживає, якщо він максимум уздовж градієнта."""
    height, width = magnitude.shape
    thin = np.zeros_like(magnitude)
    for row in range(1, height - 1):
        for col in range(1, width - 1):
            angle = angle_degrees[row, col]
            if angle < 22.5 or angle >= 157.5:          # градієнт іде вбік
                before, after = magnitude[row, col - 1], magnitude[row, col + 1]
            elif angle < 67.5:                          # градієнт іде вправо-вниз
                before, after = magnitude[row - 1, col - 1], magnitude[row + 1, col + 1]
            elif angle < 112.5:                         # градієнт іде вниз
                before, after = magnitude[row - 1, col], magnitude[row + 1, col]
            else:                                       # градієнт іде вліво-вниз
                before, after = magnitude[row - 1, col + 1], magnitude[row + 1, col - 1]
            if magnitude[row, col] >= before and magnitude[row, col] >= after:
                thin[row, col] = magnitude[row, col]
    return thin


start = time.time()
thin_edges = non_maximum_suppression(canny_magnitude, canny_angle)
print("притлумлення немаксимумів зайняло", round(time.time() - start, 2), "с")
print("пікселів із модулем > 100 було:", int((canny_magnitude > 100).sum()))
print("лишилось після притлумлення   :", int((thin_edges > 100).sum()))

**Крок 4 — гістерезис двома порогами.** Один поріг завжди поганий: підняти —
межа рветься на пунктир, опустити — вилазить шум. Кенні бере два.

Піксель із модулем **вище верхнього** порога — межа беззаперечно. Піксель між нижнім
і верхнім — межа **лише якщо** він сполучений із таким беззаперечним. Усе, що нижче
нижнього, викидається.

Сполученість шукаємо через `cv2.connectedComponents`: якщо у звʼязній групі слабких
пікселів є хоч один сильний — уся група стає межею.

In [ ]:
def hysteresis(thin, low_threshold, high_threshold):
    """Слабкий піксель стає межею, тільки якщо доросле до нього тягнеться від сильного."""
    strong = thin >= high_threshold
    candidate = thin >= low_threshold

    # 8-звʼязність: діагональні сусіди теж вважаються сполученими
    count, labels = cv2.connectedComponents((candidate * 255).astype(np.uint8),
                                            connectivity=8)
    keep = np.zeros(count, dtype=bool)
    keep[np.unique(labels[strong])] = True
    keep[0] = False                                   # мітка 0 — це фон
    return keep[labels]


LOW, HIGH = 100, 200
my_edges = hysteresis(thin_edges, LOW, HIGH)

print("сильних пікселів (модуль ≥", HIGH, "):", int((thin_edges >= HIGH).sum()))
print("слабких кандидатів (від", LOW, "до", HIGH, "):",
      int(((thin_edges >= LOW) & (thin_edges < HIGH)).sum()))
print("з них урятовано сполученістю:",
      int(my_edges.sum() - (thin_edges >= HIGH).sum()))
print("усього пікселів межі         :", int(my_edges.sum()))

### Звірка з `cv2.Canny`

Тут збігу до пікселя не буде, і це найцікавіше місце теми. Наш Кенні округлює напрямок
до чотирьох сторін, OpenCV робить це своїм способом і по-своєму розвʼязує нічиї, коли
два сусіди рівні. Тому міряємо не рівність, а **частку спільних пікселів**:
скільки пікселів обидві реалізації назвали межею, поділити на скільки їх назвала
хоча б одна.

На вхід бібліотечній функції подаємо **згладжене** зображення — те саме, з якого
починали ми. Інакше порівнювались би різні алгоритми, а не різні реалізації одного.

In [ ]:
library_edges = cv2.Canny(smoothed, LOW, HIGH, apertureSize=3, L2gradient=True) > 0

both = int((my_edges & library_edges).sum())
either = int((my_edges | library_edges).sum())
agreement = both / either * 100

print("наших пікселів межі :", int(my_edges.sum()))
print("у cv2.Canny         :", int(library_edges.sum()))
print("спільних            :", both)
print("збіг                :", round(agreement, 1), "%")

assert agreement > 85, "збіг мав бути щонайменше 85 % — інакше десь помилка в кроках"
print()
print("✅ той самий алгоритм, різниця — в дрібницях округлення напрямку")
print()
print("а якби ми забули згладити й подали сире фото, cv2.Canny знайшла б",
      int((cv2.Canny(gray, LOW, HIGH, apertureSize=3, L2gradient=True) > 0).sum()),
      "пікселів межі замість", int(library_edges.sum()))
print("зайві — це шум матриці, який Собель радо підсилив")

In [ ]:
figure, axes = plt.subplots(1, 5, figsize=(15, 2.8))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("1 · оригінал", fontsize=10)
axes[1].imshow(smoothed, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("2 · згладжене", fontsize=10)
axes[2].imshow(canny_magnitude, cmap="magma", vmin=0, vmax=300)
axes[2].set_title("3 · модуль градієнта", fontsize=10)
axes[3].imshow(thin_edges, cmap="magma", vmin=0, vmax=300)
axes[3].set_title("4 · тільки гребені", fontsize=10)
axes[4].imshow(my_edges, cmap="gray")
axes[4].set_title("5 · після гістерезису", fontsize=10)
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("між третьою й четвертою картинками межі худнуть до одного пікселя")

### Пастка: на чистій фігурі пороги не працюють

Якщо спробувати показати роль порогів на ідеальному чорному квадраті, вийде
демонстрація, у якій нічого не відбувається. Різкий перехід 0 → 255 дає модуль градієнта
близько 1082, і **будь-яка** пара порогів нижче цього числа лишає ту саму межу.

Пороги починають щось означати лише там, де є **слабкі** межі: шум, текстура, мʼякий
перехід. Порівняймо чистий квадрат і наше фото з шумом матриці.

In [ ]:
clean_square = np.zeros((120, 160), dtype=np.uint8)
clean_square[30:90, 40:120] = 255

square_gx = cv2.Sobel(clean_square.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3)
square_gy = cv2.Sobel(clean_square.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3)
print("максимальний модуль градієнта на чистому квадраті:",
      round(float(np.hypot(square_gx, square_gy).max()), 1))
print()

print(f"{'пороги':<14}{'чистий квадрат':>18}{'наше фото':>14}")
for low, high in [(20, 60), (50, 150), (100, 200), (150, 250)]:
    square_pixels = int((cv2.Canny(clean_square, low, high) > 0).sum())
    photo_pixels = int((cv2.Canny(gray, low, high) > 0).sum())
    print(f"{str(low) + ' / ' + str(high):<14}{square_pixels:>18}{photo_pixels:>14}")

print()
print("на квадраті число не змінюється жодного разу, на фото падає більш ніж утричі")

## 13 · Куди це веде

Одне ядро дає одну відповідь на піксель — і ця відповідь однобока. Ядро для вертикальних
меж не бачить горизонтальних — це ми щойно поміряли: різниця у 30 разів. Але є й друга
сліпота, про яку згадують рідше: **масштаб**. Ядро 3 × 3 дивиться на три пікселі
й міряє перепад саме на такій відстані. Перепад тієї самої висоти, але розмазаний
на сорок пікселів, воно майже не помітить.

Порахуймо це на штучних перепадах однакової висоти (з 10 до 90), розмазаних на різну
ширину.

Наступна тема бере ці відповіді й будує з них опис усього зображення — і показує,
де такий підхід ламається.

In [ ]:
def ramp_response(spread_pixels):
    """Максимум |gx| на перепаді 10 → 90, розмазаному на задану ширину."""
    columns = np.arange(60.0)
    profile = np.clip((columns - 30 + spread_pixels / 2) / max(spread_pixels, 1), 0, 1)
    profile = profile * 80 + 10
    strip = np.tile(profile, (9, 1))
    gradient_x = cv2.Sobel(strip, cv2.CV_64F, 1, 0, ksize=3)
    return float(np.abs(gradient_x[4]).max())


print(f"{'перепад розмазано на':<24}{'максимум |gx|':>16}")
for spread in (1, 3, 5, 10, 20, 40):
    print(f"{str(spread) + ' пікселів':<24}{ramp_response(spread):>16.1f}")

print()
print("той самий перепад висотою 80 рівнів дає то 320, то 16 — залежно від того,")
print("на скількох пікселях він відбувається. Одне ядро міряє один масштаб.")

---

## Завдання

### 🟢 Рівень 1
Додай у `correlate_by_hand` необовʼязковий аргумент `mode="same"`, який при значенні
`"valid"` **не** добудовує кадр, а повертає менший масив — рівно ті пікселі, де вікно
помістилось цілком.
**Зроблено, якщо** для ядра 5 × 5 на фрагменті 60 × 80 функція повертає масив 56 × 76,
і `np.allclose` звіряє його з відповідним зрізом варіанта `"same"`.

### 🟡 Рівень 2
Побудуй ядро руху під кутом 45° (одиниці по головній діагоналі, поділені на довжину)
розміром 9 × 9 і застосуй його двічі: як кореляцію й як згортку.
**Зроблено, якщо** ти показуєш два зображення поруч, називаєш словами, у який бік
змазує кожне, і числом підтверджуєш, що вони різні — наприклад, часткою пікселів,
де різниця більша за одиницю.

### 🔴 Рівень 3
Збери всі чотири кроки Кенні в одну функцію `my_canny(image, low, high)` і перевір її
на трьох парах порогів.
**Зроблено, якщо** для кожної пари збіг із `cv2.Canny(..., L2gradient=True)` за формулою
«спільні ÷ обʼєднання» не менший за **85 %**, і ти можеш пояснити, звідки береться
решта відсотків.